# 03 — VGG-style Network (from scratch)

In [1]:
import sys
import time
sys.path.insert(0, "..")
sys.path.insert(0, "../experiments")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset,
    get_train_transform, get_eval_transform, build_dataloaders,
    train_model, save_checkpoint, load_checkpoint,
    plot_training_history, print_model_info,
    compute_multilabel_metrics, evaluate_predictor,
    print_metric_table, NUM_LABELS, METRIC_KEYS,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")


Device: cuda
Labels (12): ['pen', 'paper', 'book', 'clock', 'phone', 'laptop', 'chair', 'desk', 'bottle', 'keychain', 'backpack', 'calculator']


In [2]:
BASE_DIR        = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 64
SPLIT           = [0.7, 0.15, 0.15]
CHECKPOINT_DIR  = Path("../checkpoints")
EXPERIMENT_NAME = "vgg_scratch"
MODEL_PATH      = CHECKPOINT_DIR / f"final_{EXPERIMENT_NAME}.pth"

full_dataset = load_dataset(BASE_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform,
    batch_size=BATCH_SIZE,
)
print(f"Train: {len(train_raw)}  |  Val: {len(val_raw)}  |  Test: {len(test_raw)}")


Train: 3180  |  Val: 681  |  Test: 682


## Model Definition

In [3]:
def vgg_block(in_ch, out_ch, n_convs):
    layers = []
    for i in range(n_convs):
        layers += [nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
                   nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGGScratch(nn.Module):
    """VGG-style network trained from scratch on 128x128 inputs."""
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            vgg_block(3,   64,  2),   # 128 -> 64
            vgg_block(64,  128, 2),   # 64  -> 32
            vgg_block(128, 256, 3),   # 32  -> 16
            vgg_block(256, 512, 3),   # 16  -> 8
            vgg_block(512, 512, 3),   # 8   -> 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 2048), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(2048, 512),         nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))

def create_model(num_labels):
    return VGGScratch(num_classes=num_labels)

print_model_info(create_model(NUM_LABELS))


  Total params     :   32,557,644
  Trainable params :   32,557,644  (100.0%)
  Model size       : 124.20 MB  (float32 weights)


## Grid Search (LR × WD)

In [4]:
GRID = [
    {"lr": 1e-3, "wd": 1e-4},
    {"lr": 1e-3, "wd": 1e-3},
    {"lr": 3e-4, "wd": 1e-4},
    {"lr": 1e-4, "wd": 1e-4},
]

grid_results = []
for cfg in GRID:
    print(f"\n--- lr={cfg['lr']:.0e}  wd={cfg['wd']:.0e} ---")
    state, val_f1, _, epochs_run = train_model(
        create_model, NUM_LABELS, train_loader, val_loader, DEVICE,
        lr=cfg["lr"], weight_decay=cfg["wd"],
        max_epochs=20, warmup_epochs=2, early_stop_patience=5,
    )
    grid_results.append({**cfg, "val_f1": val_f1, "state": state, "epochs": epochs_run})
    print(f"  => val F1: {val_f1:.4f}")

grid_results.sort(key=lambda x: x["val_f1"], reverse=True)
best = grid_results[0]
print(f"\nBest config: lr={best['lr']:.0e}  wd={best['wd']:.0e}  val_F1={best['val_f1']:.4f}")

rows = [{"lr": c["lr"], "wd": c["wd"], "val_f1": round(c["val_f1"], 4), "epochs": c["epochs"]}
        for c in grid_results]
print(pd.DataFrame(rows).to_string(index=False))



--- lr=1e-03  wd=1e-04 ---

Epoch  1/20  [lr=5.00e-04]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.6811    0.4185
  exact_match            0.0063    0.0000
  hamming_acc            0.8200    0.8532
  mean_iou               0.0269    0.0000
  precision_micro        0.1420    0.0000
  recall_micro           0.0450    0.0000
  f1_micro               0.0684    0.0000

Epoch  2/20  [lr=1.00e-03]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.4958    0.4242
  exact_match            0.0025    0.0000
  hamming_acc            0.8452    0.8532
  mean_iou               0.0104    0.0000
  precision_micro        0.1612    0.0000
  recall_micro           0.0132    0.0000
  f1_micro               0.0244    0.0000

Epoch  3/20  [lr=1.00e-03]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.4531    0.4189
  e

## Final Training

In [ ]:
t0 = time.time()
best_state, best_val_f1, history, epochs_run = train_model(
    create_model, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=best["lr"], weight_decay=best["wd"],
    max_epochs=60, warmup_epochs=5, early_stop_patience=10,
)
training_time = time.time() - t0
print(f"\nBest val F1: {best_val_f1:.4f}  |  Epochs: {epochs_run}  |  Time: {training_time:.1f}s")

save_checkpoint(best_state, MODEL_PATH)
plot_training_history(history, epochs_run, EXPERIMENT_NAME, best["lr"], best["wd"])



Epoch  1/60  [lr=2.00e-05]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.6443    0.4220
  exact_match            0.0170    0.0000
  hamming_acc            0.7792    0.8532
  mean_iou               0.0722    0.0000
  precision_micro        0.1585    0.0000
  recall_micro           0.1174    0.0000
  f1_micro               0.1349    0.0000

Epoch  2/60  [lr=4.00e-05]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.4876    0.4166
  exact_match            0.0157    0.0000
  hamming_acc            0.8248    0.8533
  mean_iou               0.0492    0.0007
  precision_micro        0.1930    1.0000
  recall_micro           0.0613    0.0008
  f1_micro               0.0930    0.0017

Epoch  3/60  [lr=6.00e-05]
  Metric                  Train       Val
  ----------------------------------------
  loss                   0.4626    0.4092
  exact_match            0.0217

## Evaluation

In [ ]:
model = load_checkpoint(create_model, NUM_LABELS, MODEL_PATH, DEVICE)
model.eval()

def _predict(images, threshold=0.5):
    with torch.no_grad():
        logits = model(images)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= threshold).float()
    return preds, probs, logits

val_metrics  = evaluate_predictor(val_loader,  _predict, DEVICE)
test_metrics = evaluate_predictor(test_loader, _predict, DEVICE)

rows = [
    {"split": "val",  **{k: round(val_metrics[k],  4) for k in METRIC_KEYS}},
    {"split": "test", **{k: round(test_metrics[k], 4) for k in METRIC_KEYS}},
]
df = pd.DataFrame(rows).set_index("split")
print(df.to_string())


In [ ]:

print("\nModel summary:")
print_model_info(create_model(NUM_LABELS))
print(f"Training time : {training_time:.1f}s")
